In [0]:
from pyspark.sql import functions as F
pred = spark.table("h2.gold.day6_classifier_predictions") \
    .withColumn("run_id", F.concat(F.lit("day8_"), F.date_format(F.current_timestamp(), "yyyyMMdd_HHmmss")))
# pred.write.mode("overwrite").format("delta").saveAsTable("h2.gold.refuel_risk_predictions")
# spark.table("h2.gold.refuel_risk_predictions").selectExpr(
#     "count(*) rows", "min(event_time_utc) min_ts", "max(event_time_utc) max_ts"
# ).show(truncate=False)

top20_overall = pred.orderBy(F.desc("score_positive")).limit(20)
display(top20_overall)

In [0]:
from pyspark.sql.window import Window
pred_day = pred.withColumn("event_date", F.to_date("event_time_utc"))

w = Window.partitionBy("event_date").orderBy(F.desc("score_positive"))

top5_per_day = (
    pred_day.withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 5)
    .orderBy("event_date", "rank")
)
display(top5_per_day)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS

ratings = (
    spark.table("h2.gold.refuel_risk_predictions")
    .selectExpr(
        "cast(abs(hash(zone)) %100000 as int) as user_id",
        "cast(hour(event_time_utc) as int) as item_id",
        "cast(score_positive as double) as rating"
    )
    .filter("rating is not null")
)
als = ALS(userCol="user_id", itemCol="item_id", ratingCol="rating",
          rank=20, maxIter=10, regParam=0.1, nonnegative=True,
          coldStartStrategy="drop")
als_model = als.fit(ratings)

top5 = als_model.recommendForAllUsers(5)

In [0]:
# # Workaround: Collect a small number of ALS results to the driver for inspection.
# # Avoids explode, posexplode, and saveAsTable, so is compatible with Unity Catalog on serverless.
# # Only suitable for small samples for inspection.

# als_sample = top5.limit(10).collect()
# for row in als_sample:
#     print(f"user_id: {row['user_id']}")
#     for rec in row['recommendations']:
#         print(f"    item_id: {rec.item_id}, rating: {rec.rating}")
#     print('-' * 20)

In [0]:
# top5.write.mode("overwrite").format("delta").saveAsTable("h2.gold.top5_recommendations")
# display(top5.limit(20))

In [0]:
# parquet_path = '/tmp/top5_recommendations.parquet'
# top5.write.mode('overwrite').parquet(parquet_path)
# print(f"ALS recommendations saved to: {parquet_path}")
# try:
#     dbutils.fs.ls(parquet_path)
# except Exception as e:
#     print("File written; see /tmp/top5_recommendations.parquet in workspace Files sidebar.")

In [0]:
import time
q = "SELECT zone, hour(event_time_utc) hr, avg(score_positive) avg_risk FROM h2.gold.refuel_risk_predictions GROUP BY zone, hour(event_time_utc)"
print("Explain plan:")
spark.sql("EXPLAIN FORMATTED " + q).show(truncate=False)
t0 = time.time()
spark.sql(q).count()
t1 = time.time()
print("Runtime without cache:", round(t1 - t0, 2), "sec")
spark.sql("CACHE TABLE h2.gold.refuel_risk_predictions")
t2 = time.time()
spark.sql(q).count()
t3 = time.time()
print("Runtime with cache:", round(t3 - t2, 2), "sec")
spark.sql("OPTIMIZE h2.gold.refuel_risk_predictions ZORDER BY (zone, event_time_utc)")

In [0]:
%sql
-- SQL cell
DESCRIBE HISTORY h2.gold.refuel_risk_predictions;

In [0]:

from pyspark.sql import functions as F
cur = spark.table("h2.gold.refuel_risk_predictions")
new_rows = cur.limit(100).withColumn("run_id", F.lit("day11_append"))
new_rows.write.mode("append").format("delta").saveAsTable("h2.gold.refuel_risk_predictions")

In [0]:
%sql
-- SQL cell: compare current vs previous version
SELECT COUNT(*) AS current_rows FROM h2.gold.refuel_risk_predictions;
SELECT COUNT(*) AS old_rows FROM h2.gold.refuel_risk_predictions VERSION AS OF 0;
---

In [0]:
tables = [
    "h2.silver.model_training_day",
    "h2.silver.model_training",
    "h2.silver.model_test",
    "h2.gold.refuel_risk_predictions"
]
for t in tables:
    print("\nTable:", t)
    spark.sql(f"DESCRIBE DETAIL {t}").select("format","numFiles","sizeInBytes").show(truncate=False)

In [0]:
from pyspark.sql import functions as F
# Final pipeline health check
checks = {
    "h2.silver.model_training_day": spark.table("h2.silver.model_training_day").count(),
    "h2.silver.model_training": spark.table("h2.silver.model_training").count(),
    "h2.silver.model_test": spark.table("h2.silver.model_test").count(),
    "h2.gold.day6_model_leaderboard": spark.table("h2.gold.day6_model_leaderboard").count(),
    "h2.gold.refuel_risk_predictions": spark.table("h2.gold.refuel_risk_predictions").count(),
}
print(checks)
# Ops log
spark.sql("""
CREATE SCHEMA IF NOT EXISTS h2.ops
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS h2.ops.pipeline_run_log1 (
  run_ts_utc TIMESTAMP,
  stage STRING,
  status STRING,
  row_count BIGINT
) USING DELTA
""")
rows = spark.table("h2.gold.refuel_risk_predictions").count()
spark.sql(f"""
INSERT INTO h2.ops.pipeline_run_log1
SELECT current_timestamp(), 'day14_final_pipeline', 'success', {rows}
""")